In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
import mofax as mfx
import duckdb as db
def limpiar_drug(s):
    return (s.str.strip()
             .str.upper()
             .str.replace(' ', '_', regex=False)  
             .str.replace(r'_+', '_', regex=True)
             .str.strip('_'))

In [ ]:
# --- PATHS ---
MOFA_MODEL = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/modelo_mofa_30factors.hdf5'
INPUT_PARQUET = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final.parquet"
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
OUTPUT_H5AD = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad'

# 1. Load MOFA model
print("Loading MOFA model...")
model = mfx.mofa_model(MOFA_MODEL)

# 2. Factors (samples x factors)
Z = model.get_factors(df=True)
print(f"Factors: {Z.shape}")
print(f"Example index: {Z.index[:3].tolist()}")

# 3. Metadata from the factors index
# sample format: drug_concentration_plate (plate is the last segment after _)
split = Z.index.to_series().str.rsplit('_', n=2, expand=True)
plate = split[2]
concentration = split[1]
drug = split[0]
obs = pd.DataFrame({
    'drug': clean_drug(drug).values,
    'concentration': concentration.values,
    'plate': plate.values
}, index=Z.index)

# 4. Add MOA from drug metadata
print("Adding MOA...")
drug_meta = pd.read_parquet(DRUG_PARQUET)
drug_meta['drug'] = clean_drug(drug_meta['drug'])

if 'moa-fine' in drug_meta.columns:
    moa_map = drug_meta[['drug', 'moa-fine', 'moa-broad']].drop_duplicates('drug')
    obs = obs.merge(moa_map, on='drug', how='left')
    obs.index = Z.index
    print(f"  MOA added. NaN: {obs['moa-fine'].isna().sum()}")
else:
    print(f"  Columns available in drug_meta: {drug_meta.columns.tolist()}")
    print("  Adjust the MOA column name manually")

# 5. MOFA weights
print("Extracting MOFA weights...")

# 6. Create AnnData
print("Creating AnnData...")
adata = ad.AnnData(
    X=Z.values,
    obs=obs,
    var=pd.DataFrame(index=Z.columns),
)

# 7. Save weights in uns — correct average across views (cell lines)
views = list(model.get_views())
W_per_view = {view: model.get_weights(views=view, df=True) for view in views}

# Save individual weights per view
for view, w in W_per_view.items():
    adata.uns[f'mofa_weights_{view}'] = w.values
    adata.uns[f'mofa_weights_genes_{view}'] = w.index.tolist()
adata.uns['mofa_weights_factors'] = list(Z.columns)
adata.uns['mofa_views'] = views

# Average across views aligning genes → consensus matrix for decoupler
genes_ref = list(W_per_view.values())[0].index
W_array = np.stack(
    [w.reindex(genes_ref).values for w in W_per_view.values()], axis=0
)  # (n_views, n_genes, n_factors)
W_mean = np.nanmean(W_array, axis=0)  # ignore NaN in the average

# Now we can remove genes that are NaN across ALL views
nan_in_all = np.isnan(W_array).all(axis=0).all(axis=1)  # genes with no data in any view
print(f"Genes with no data in any view: {nan_in_all.sum()}")

W_mean = W_mean[~nan_in_all]
genes_ref_clean = genes_ref[~nan_in_all]

adata.uns['mofa_weights'] = W_mean
adata.uns['mofa_weights_genes'] = genes_ref_clean.tolist()

print(f"Consensus weights: {W_mean.shape} ({len(genes_ref)} genes x {W_mean.shape[1]} factors)")
print(f"Rows with all zeros: {(W_mean == 0).all(axis=1).sum()}")

# 8. Clean object columns with NaN before saving
for col in adata.obs.columns:
    if adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].fillna('unknown').astype(str)

# 9. Save
print(f"\n{adata}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")
adata.write(OUTPUT_H5AD, compression='gzip')
print(f"Saved to {OUTPUT_H5AD}")